In [1]:
#!pip install jupyter pandas yfinance aiohttp polars polars-talib==0.1.5 cryptography

In [2]:
import logging
from datetime import datetime, date, timedelta
from typing import Union, Optional

logger = logging.getLogger(__name__)

class DateUtils:
    """Utility functions for date handling"""
    
    @staticmethod
    def convert_to_datetime(date_input: Union[str, datetime, date]) -> datetime:
        """Convert various date formats to datetime object"""
        if isinstance(date_input, datetime):
            return date_input
        elif isinstance(date_input, date):
            return datetime.combine(date_input, datetime.min.time())
        elif isinstance(date_input, str):
            try:
                # Try parsing as ISO format first
                if 'T' in date_input or 'Z' in date_input:
                    # Handle ISO format with timezone
                    date_input = date_input.replace('Z', '+00:00')
                    return datetime.fromisoformat(date_input)
                else:
                    # Handle simple date format (YYYY-MM-DD)
                    return datetime.strptime(date_input, '%Y-%m-%d')
            except ValueError as e:
                logger.error(f"Error parsing date string '{date_input}': {e}")
                raise ValueError(f"Invalid date format: {date_input}. Expected YYYY-MM-DD or ISO format")
        else:
            raise ValueError(f"Unsupported date type: {type(date_input)}. Expected str, datetime, or date")
    
    @staticmethod
    def format_date_for_api(date_obj: Union[datetime, date]) -> str:
        """Format date for API requests"""
        if isinstance(date_obj, datetime):
            return date_obj.isoformat() + 'Z'
        elif isinstance(date_obj, date):
            return datetime.combine(date_obj, datetime.min.time()).isoformat() + 'Z'
        else:
            raise ValueError(f"Unsupported date type: {type(date_obj)}")
    
    @staticmethod
    def validate_date_range(start_date: Union[str, datetime, date], 
                          end_date: Union[str, datetime, date]) -> bool:
        """Validate that start_date is before end_date"""
        start_dt = DateUtils.convert_to_datetime(start_date)
        end_dt = DateUtils.convert_to_datetime(end_date)
        
        if start_dt >= end_dt:
            raise ValueError("Start date must be before end date")
        
        return True
    
    @staticmethod
    def get_trading_days(start_date: Union[str, datetime, date], 
                        end_date: Union[str, datetime, date]) -> int:
        """Get number of trading days between dates (approximate)"""
        start_dt = DateUtils.convert_to_datetime(start_date)
        end_dt = DateUtils.convert_to_datetime(end_date)
        
        # Simple calculation (doesn't account for holidays/weekends)
        delta = end_dt - start_dt
        return delta.days
    
    @staticmethod
    def is_market_hours(dt: datetime) -> bool:
        """Check if datetime is during market hours (9:30 AM - 4:00 PM ET)"""
        # Convert to Eastern Time (simplified)
        # In production, you'd use pytz for proper timezone handling
        hour = dt.hour
        minute = dt.minute
        
        # Market hours: 9:30 AM - 4:00 PM ET
        market_start = 9 * 60 + 30  # 9:30 AM in minutes
        market_end = 16 * 60  # 4:00 PM in minutes
        current_time = hour * 60 + minute
        
        return market_start <= current_time <= market_end
    
    @staticmethod
    def get_next_market_open(dt: datetime) -> datetime:
        """Get the next market open time"""
        # Simplified implementation
        # In production, you'd account for weekends and holidays
        if dt.weekday() >= 5:  # Saturday or Sunday
            # Move to next Monday
            days_ahead = 7 - dt.weekday()
            dt = dt + timedelta(days=days_ahead)
        
        # Set to 9:30 AM
        return dt.replace(hour=9, minute=30, second=0, microsecond=0) 

In [3]:
import asyncio
import logging
import polars as pl
from datetime import datetime, timedelta, date
from typing import List, Dict, Any, Optional
from motor.motor_asyncio import AsyncIOMotorDatabase

from data_providers import DataProviderFactory, BaseDataProvider, TIMEFRAME_MAPPINGS

logger = logging.getLogger(__name__)

class DataManager:
    """Handles all data fetching and caching using DataProviderFactory"""
    
    def __init__(self, db: AsyncIOMotorDatabase):
        self.db = db
        self.data_cache = {}
        self.data_provider = None
    
    async def initialize_provider(self, data_provider_name: str, user_id: str):
        """Initialize data provider using DataProviderFactory"""
        try:
            # Get user configuration for API keys only
            user_config = await self.db['user_config'].find_one({"user_id": user_id})
            
            if data_provider_name.lower() == 'yahoo':
                self.data_provider = DataProviderFactory.get_provider('yahoo')
                logger.info("Initialized Yahoo Finance data provider")
                
            elif data_provider_name.lower() == 'alpaca':
                if not user_config:
                    logger.warning(f"No user configuration found for user {user_id}, falling back to Yahoo Finance")
                    self.data_provider = DataProviderFactory.get_provider('yahoo')
                    return
                
                # Get API keys from user config
                api_key = user_config.get('alpaca_paper_api_key') or user_config.get('alpaca_live_api_key')
                secret_key = user_config.get('alpaca_paper_secret_key') or user_config.get('alpaca_live_secret_key')
                
                if api_key and secret_key:
                    # Decrypt secret key if needed
                    try:
                        from models.user_config import ConfigEncryption
                        decrypted_secret = ConfigEncryption.decrypt_value(secret_key)
                        logger.info(f"DEBUG DATA MANAGER: Decrypted Alpaca secret key: {decrypted_secret}")
                    except Exception as e:
                        logger.warning(f"Could not decrypt secret key, using as-is: {e}")
                        decrypted_secret = secret_key
                        
                    self.data_provider = DataProviderFactory.get_provider(
                        'alpaca',
                        api_key=api_key,
                        secret_key=decrypted_secret
                    )
                    logger.info("Initialized Alpaca data provider")
                else:
                    logger.warning(f"Alpaca API keys not found for user {user_id}, falling back to Yahoo Finance")
                    self.data_provider = DataProviderFactory.get_provider('yahoo')
                    
            elif data_provider_name.lower() == 'polygon':
                if not user_config:
                    logger.warning(f"No user configuration found for user {user_id}, falling back to Yahoo Finance")
                    self.data_provider = DataProviderFactory.get_provider('yahoo')
                    return
                
                # Get the actual API key (polygon_secret_key), not the key name
                api_key = user_config.get('polygon_secret_key')
                
                if api_key:
                    # Decrypt the API key if needed
                    try:
                        from models.user_config import ConfigEncryption
                        decrypted_api_key = ConfigEncryption.decrypt_value(api_key)
                        logger.info(f"DEBUG DATA MANAGER: Decrypted Polygon API key: {decrypted_api_key}")
                    except Exception as e:
                        logger.warning(f"Could not decrypt Polygon API key, using as-is: {e}")
                        decrypted_api_key = api_key
                    
                    self.data_provider = DataProviderFactory.get_provider('polygon', api_key=decrypted_api_key)
                    logger.info("Initialized Polygon data provider")
                else:
                    logger.warning(f"Polygon API key not found for user {user_id}, falling back to Yahoo Finance")
                    self.data_provider = DataProviderFactory.get_provider('yahoo')
                    
            else:
                logger.warning(f"Unknown data provider '{data_provider_name}', falling back to Yahoo Finance")
                self.data_provider = DataProviderFactory.get_provider('yahoo')
                
        except Exception as e:
            logger.error(f"Error initializing data provider: {e}, falling back to Yahoo Finance")
            self.data_provider = DataProviderFactory.get_provider('yahoo')
    
    async def fetch_historical_data(self, symbols: List[str], start_date: str, end_date: str, timeframe: str, data_provider: str) -> pl.DataFrame:
        """Fetch historical data for symbols using the initialized provider"""
        cache_key = f"{'-'.join(symbols)}_{start_date}_{end_date}_{timeframe}_{self.data_provider.get_provider_name()}"
        
        if cache_key in self.data_cache:
            logger.info(f"Using cached data for {symbols}")
            return self.data_cache[cache_key]
        
        logger.info(f"Fetching data for {symbols} from {start_date} to {end_date} using {self.data_provider.get_provider_name()}")
        
        try:
            start_dt = self._convert_to_datetime(start_date)
            end_dt = self._convert_to_datetime(end_date)
            
            all_data = []
            for symbol in symbols:
                try:
                    data = await self.data_provider.get_historical_data(
                        symbol=symbol,
                        start_date=start_dt,
                        end_date=end_dt,
                        timeframe=timeframe,
                        data_provider=data_provider
                    )
                    
                    # Fix: Check if Polars DataFrame is empty using height
                    if data.height > 0:  # Polars uses .height instead of .empty
                        # Add symbol column if it doesn't exist
                        if 'symbol' not in data.columns:
                            data = data.with_columns(pl.lit(symbol).alias("symbol"))
                        all_data.append(data)
                        logger.info(f"Retrieved {data.height} data points for {symbol}")
                    else:
                        logger.warning(f"No data retrieved for {symbol}")
                        
                except Exception as e:
                    logger.error(f"Error fetching data for {symbol}: {e}")
                    continue
            
            if not all_data:
                raise ValueError("No data retrieved for any symbols")
            
            combined_data = pl.concat(all_data, how="vertical")
            combined_data = self._standardize_columns(combined_data)
            combined_data = combined_data.sort(["datetime", "symbol"])
            
            self.data_cache[cache_key] = combined_data
            logger.info(f"Retrieved {len(combined_data)} total data points")
            return combined_data
            
        except Exception as e:
            logger.error(f"Error fetching data: {e}")
            raise
    
    async def fetch_data(self, symbols: List[str], timeframe: str, limit: int, data_provider: str) -> pl.DataFrame:
        """
        Fetches the latest market data for a set of symbols, up to a specified limit.

        This function calculates a suitable start date to ensure enough data is retrieved
        to satisfy the limit, accounting for non-trading days.
        """
        logger.info(f"Fetching latest {limit} data points for {symbols} with timeframe {timeframe} using {data_provider}")

        end_date = datetime.now()
        
        # Get the timeframe category and calculate lookback period
        days_to_look_back = self._calculate_lookback_days(timeframe, limit)

        start_date = end_date - timedelta(days=max(1, days_to_look_back))
        
        start_date_str = start_date.strftime('%Y-%m-%d')
        end_date_str = end_date.strftime('%Y-%m-%d')

        symbols = [symbol.upper() for symbol in symbols]

        try:
            historical_data = await self.fetch_historical_data(
                symbols=symbols,
                start_date=start_date_str,
                end_date=end_date_str,
                timeframe=timeframe,
                data_provider=data_provider
            )

            if historical_data.height == 0:
                return historical_data

            # Ensure we only return the last `limit` data points per symbol
            return historical_data.group_by('symbol', maintain_order=True).tail(limit)

        except Exception as e:
            logger.error(f"Failed to fetch latest market data: {e}")
            # Return an empty DataFrame on failure to prevent crashes downstream
            return pl.DataFrame()

    def _calculate_lookback_days(self, timeframe: str, limit: int) -> int:
        """
        Calculate the number of days to look back based on timeframe and limit.
        Uses the TIMEFRAME_MAPPINGS to categorize timeframes properly.
        """
        from .data_providers import TIMEFRAME_MAPPINGS
        
        # Normalize timeframe to match our mappings
        normalized_timeframe = self._normalize_timeframe(timeframe)
        
        if normalized_timeframe not in TIMEFRAME_MAPPINGS:
            logger.warning(f"Unknown timeframe format: {timeframe}. Using daily assumption for lookback calculation.")
            return int(limit * 1.8)  # Default assumption: ~7 trading days in 10 calendar days
        
        # Determine timeframe category based on the normalized timeframe
        if normalized_timeframe in ['1M', '2M', '5M', '15M', '30M']:
            # Minute-based timeframes
            minutes = self._extract_minutes_from_timeframe(normalized_timeframe)
            # Assume 6.5 trading hours per day (390 minutes)
            trading_days_needed = (limit * minutes) / 390
            # Add buffer for weekends, holidays, and market closures
            return int(trading_days_needed * 2.5) + 5
            
        elif normalized_timeframe in ['60M', '1H']:
            # Hour-based timeframes
            hours = 1  # Both 60M and 1H represent 1 hour
            # Assume 6.5 trading hours per day
            trading_days_needed = (limit * hours) / 6.5
            return int(trading_days_needed * 2.5) + 5
            
        elif normalized_timeframe in ['1d', '1D']:
            # Daily timeframes
            return int(limit * 1.8)  # ~7 trading days in 10 calendar days
            
        elif normalized_timeframe in ['1wk', '1w', '1W']:
            # Weekly timeframes
            return limit * 10  # Assume ~10 calendar days per trading week
            
        elif normalized_timeframe in ['1mo', '3mo']:
            # Monthly timeframes
            return limit * 35  # Assume ~35 days per month
            
        else:
            # Fallback for any other timeframes
            logger.warning(f"Unhandled timeframe category for: {timeframe}. Using daily assumption.")
            return int(limit * 1.8)

    def _normalize_timeframe(self, timeframe: str) -> str:
        """
        Normalize various timeframe formats to match TIMEFRAME_MAPPINGS keys.
        """
        # Handle common variations
        timeframe_upper = timeframe.upper()
        
        # Map common variations to our standard format
        variations = {
            '1MIN': '1M',
            '5MIN': '5M', 
            '15MIN': '15M',
            '30MIN': '30M',
            '60MIN': '60M',
            '1HOUR': '1H',
            '1DAY': '1D',
            '1WEEK': '1W',
            '1MONTH': '1mo',
            # Add more variations as needed
        }
        
        if timeframe_upper in variations:
            return variations[timeframe_upper]
        
        # If it's already in our mapping, return as-is
        if timeframe in TIMEFRAME_MAPPINGS or timeframe_upper in TIMEFRAME_MAPPINGS:
            return timeframe if timeframe in TIMEFRAME_MAPPINGS else timeframe_upper
        
        # Try lowercase version
        timeframe_lower = timeframe.lower()
        if timeframe_lower in TIMEFRAME_MAPPINGS:
            return timeframe_lower
            
        return timeframe  # Return original if no mapping found

    def _extract_minutes_from_timeframe(self, timeframe: str) -> int:
        """Extract the number of minutes from minute-based timeframes."""
        minute_mappings = {
            '1M': 1,
            '2M': 2,
            '5M': 5,
            '15M': 15,
            '30M': 30,
            '60M': 60
        }
        return minute_mappings.get(timeframe, 1)

    def _convert_to_datetime(self, date_input) -> datetime:
        """Convert various date formats to datetime"""
        if isinstance(date_input, datetime):
            return date_input
        elif isinstance(date_input, date):  # Add support for date objects
            return datetime.combine(date_input, datetime.min.time())
        elif isinstance(date_input, str):
            # Try different date formats
            for fmt in ['%Y-%m-%d', '%Y-%m-%dT%H:%M:%S', '%Y-%m-%dT%H:%M:%SZ']:
                try:
                    return datetime.strptime(date_input, fmt)
                except ValueError:
                    continue
            raise ValueError(f"Unable to parse date string: {date_input}")
        else:
            raise ValueError(f"Unsupported date type: {type(date_input)}")
    
    def _standardize_columns(self, df: pl.DataFrame) -> pl.DataFrame:
        """Standardize column names across different providers"""
        column_mapping = {
            'Date': 'datetime', 'Datetime': 'datetime', 
            'date': 'datetime', 'time': 'datetime', 
            'timestamp': 'datetime', 't': 'datetime',
            'Open': 'open', 'o': 'open', 'open': 'open', 'high': 'high', 
            'High': 'high', 'h': 'high', 'Low': 'low', 'l': 'low', 'low': 'low', 
            'Close': 'close', 'c': 'close', 'close': 'close', 
             'volume': 'volume', 'Volume': 'volume', 'v': 'volume',
        }
        
        existing_cols = df.columns
        rename_dict = {col: column_mapping[col] for col in existing_cols if col in column_mapping}
        if rename_dict:
            df = df.rename(rename_dict)
        
        required_columns = ['datetime', 'open', 'high', 'low', 'close', 'volume', 'symbol']
        for col in required_columns:
            if col not in df.columns:
                if col == 'datetime':
                    df = df.with_columns(pl.arange(0, len(df)).alias('datetime'))
                else:
                    df = df.with_columns(pl.lit(0).alias(col))
        
        return df
    
    def get_supported_timeframes(self) -> List[str]:
        """Get supported timeframes for the current data provider"""
        if self.data_provider is None:
            return []
        provider_name = self.data_provider.get_provider_name()
        supported = DataProviderFactory.get_supported_timeframes(provider_name)
        return supported.get(provider_name, [])
    
    def validate_timeframe(self, timeframe: str) -> bool:
        """Validate if a timeframe is supported by the current data provider"""
        if self.data_provider is None:
            return False
        supported_timeframes = self.get_supported_timeframes()
        return timeframe in supported_timeframes 

In [4]:
from datetime import datetime
from data_providers import DataProviderFactory
import sys
sys.path.append('C:/Users/fwmac/Documents/Projects/Bot-Club/app/backend_services/src/models/')
from user_config import ConfigEncryption
from cryptography.fernet import Fernet


In [5]:

data_provider_client = DataProviderFactory.get_provider(provider_name='polygon', api_key='8XvjiWID5oKnXHWl2O0n7Pc6iyAoyKG3')

start_date = datetime.strptime('2025-01-02', '%Y-%m-%d')
end_date = datetime.strptime('2025-01-10', '%Y-%m-%d')

In [6]:
pltr_async = data_provider_client.get_historical_data(symbol='PLTR', 
                                         start_date=start_date, 
                                         end_date=end_date,
                                         timeframe='15m')


pltr_data = await pltr_async
pltr_data.head()

t,open,high,low,close,volume,vwap
"datetime[ms, America/New_York]",f64,f64,f64,f64,f64,f64
2025-01-02 00:00:00 EST,76.2,76.53,72.42,75.19,7.2217192e7,74.488
2025-01-03 00:00:00 EST,75.39,79.98,75.19,79.89,6.2415008e7,78.6387
2025-01-06 00:00:00 EST,78.69,80.06,74.61,75.92,1.05619544e8,76.9685
2025-01-07 00:00:00 EST,75.2,75.39,69.75,69.99,9.1727347e7,71.62
2025-01-08 00:00:00 EST,68.12,69.53,66.51,68.23,9.1341388e7,68.0404


In [7]:
alpaca_client = DataProviderFactory.get_provider(provider_name='alpaca', 
                                                        api_key='PKW94MLMTLDYGACJ53PJ',
                                                        secret_key='Ewzw0ngdmRtT0W4R1JqTVBOfezkEq5d92SKIznOO')


In [8]:
pltr_alpaca_async = alpaca_client.get_historical_data(symbol='PLTR', 
                                         start_date=start_date, 
                                         end_date=end_date,
                                         timeframe='15m')

pltr_alpaca = await pltr_alpaca_async
pltr_alpaca.head()

t,open,high,low,close,volume,vwap
"datetime[μs, America/New_York]",f64,f64,f64,f64,i64,f64
2025-01-02 00:00:00 EST,76.2,76.53,72.42,75.19,72217192,74.427641
2025-01-03 00:00:00 EST,75.39,79.98,75.19,79.89,62415008,78.666786
2025-01-06 00:00:00 EST,78.69,80.06,74.61,75.92,105619544,76.923504
2025-01-07 00:00:00 EST,75.2,75.39,69.75,69.99,91727347,71.592514
2025-01-08 00:00:00 EST,68.12,69.53,66.51,68.23,91349513,68.031506


In [9]:
for row in pltr_alpaca.iter_rows(named=True):
    print(row, '\n')

{'t': datetime.datetime(2025, 1, 2, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')), 'open': 76.2, 'high': 76.53, 'low': 72.42, 'close': 75.19, 'volume': 72217192, 'vwap': 74.427641} 

{'t': datetime.datetime(2025, 1, 3, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')), 'open': 75.39, 'high': 79.98, 'low': 75.19, 'close': 79.89, 'volume': 62415008, 'vwap': 78.666786} 

{'t': datetime.datetime(2025, 1, 6, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')), 'open': 78.69, 'high': 80.06, 'low': 74.61, 'close': 75.92, 'volume': 105619544, 'vwap': 76.923504} 

{'t': datetime.datetime(2025, 1, 7, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')), 'open': 75.2, 'high': 75.39, 'low': 69.75, 'close': 69.99, 'volume': 91727347, 'vwap': 71.592514} 

{'t': datetime.datetime(2025, 1, 8, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')), 'open': 68.12, 'high': 69.53, 'low': 66.51, 'close': 68.23, 'volume': 91349513, 'vwap': 68.031506} 



In [10]:
row

{'t': datetime.datetime(2025, 1, 8, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')),
 'open': 68.12,
 'high': 69.53,
 'low': 66.51,
 'close': 68.23,
 'volume': 91349513,
 'vwap': 68.031506}

In [11]:
row[
    'close'
]

68.23

In [12]:
type(row)

dict

In [13]:
asset_list = await alpaca_client.get_asset_list()

In [14]:
asset_df = pl.DataFrame(asset_list)

In [15]:
crypto_assets = asset_df.filter(pl.col('exchange')=='CRYPTO')


In [16]:
unique_categories = crypto_assets["symbol"].unique().to_list()
unique_categories


['BAT/USDC',
 'BTC/USD',
 'TRUMP/USD',
 'AAVE/USDC',
 'SOL/USD',
 'UNI/USD',
 'MKR/USD',
 'YFI/USDC',
 'XTZ/USDC',
 'YFI/USDT',
 'SHIB/USD',
 'AVAX/USDT',
 'YFI/USD',
 'AVAX/USDC',
 'LINK/USDC',
 'BTC/USDT',
 'SUSHI/USDT',
 'CRV/USD',
 'LTC/USDT',
 'AVAX/USD',
 'SOL/USDC',
 'USDG/USD',
 'USDT/USDC',
 'DOT/USD',
 'BCH/BTC',
 'ETH/BTC',
 'AAVE/USD',
 'UNI/BTC',
 'DOT/USDC',
 'BCH/USD',
 'BTC/USDC',
 'BCH/USDT',
 'BCH/USDC',
 'DOGE/USDC',
 'SUSHI/USD',
 'XTZ/USD',
 'PEPE/USD',
 'LINK/USDT',
 'SOL/USDT',
 'LINK/BTC',
 'XRP/USD',
 'AAVE/USDT',
 'CRV/USDC',
 'DOGE/USDT',
 'ETH/USDC',
 'LTC/USD',
 'UNI/USDT',
 'SHIB/USDC',
 'GRT/USDC',
 'LINK/USD',
 'ETH/USD',
 'ETH/USDT',
 'SUSHI/USDC',
 'MKR/USDC',
 'DOGE/USD',
 'UNI/USDC',
 'USDT/USD',
 'LTC/BTC',
 'GRT/USD',
 'USDC/USD',
 'LTC/USDC',
 'SHIB/USDT',
 'BAT/USD']

In [56]:
alpaca_crypto_list = set(sorted([x.rstrip('/USD') for x in unique_categories if x.endswith('/USD')]))
alpaca_crypto_list

{'AAVE',
 'AVAX',
 'BAT',
 'BCH',
 'BTC',
 'CRV',
 'DOGE',
 'DOT',
 'ETH',
 'GRT',
 'LINK',
 'LTC',
 'MKR',
 'PEPE',
 'SHIB',
 'SOL',
 'SUSHI',
 'TRUMP',
 'UNI',
 'USDC',
 'USDG',
 'USDT',
 'XRP',
 'XTZ',
 'YFI'}

In [48]:
polygon_crypto_assets = '''{"results":[{"ticker":"X:00USD","name":"00 Token - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"00","base_currency_name":"00 Token","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:1INCHUSD","name":"1inch - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"1INCH","base_currency_name":"1inch","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:A8USD","name":"Ancient8 - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"A8","base_currency_name":"Ancient8","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AAVEUSD","name":"Aave - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AAVE","base_currency_name":"Aave","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ABTUSD","name":"Arcblock - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ABT","base_currency_name":"Arcblock","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ACAUSD","name":"Acala - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ACA","base_currency_name":"Acala","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ACHUSD","name":"Alchemy Pay - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ACH","base_currency_name":"Alchemy Pay","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ACSUSD","name":"Access Protocol - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ACS","base_currency_name":"Access Protocol","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ACXUSD","name":"Across Protocol - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ACX","base_currency_name":"Across Protocol","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ADAUSD","name":"Cardano - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ADA","base_currency_name":"Cardano","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ADXUSD","name":"AdEx - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ADX","base_currency_name":"AdEx","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AERGOUSD","name":"Aergo - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AERGO","base_currency_name":"Aergo","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AEROUSD","name":"Aerodome Finance - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AERO","base_currency_name":"Aerodome Finance","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AEVOUSD","name":"Aevo - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AEVO","base_currency_name":"Aevo","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AGLDUSD","name":"Adventure Gold - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AGLD","base_currency_name":"Adventure Gold","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AIOZUSD","name":"AIOZ Network - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AIOZ","base_currency_name":"AIOZ Network","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AIRUSD","name":"AIRian - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AIR","base_currency_name":"AIRian","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AKTEUR","name":"Akash Network - Euro","market":"crypto","locale":"global","active":true,"currency_symbol":"EUR","currency_name":"Euro","base_currency_symbol":"AKT","base_currency_name":"Akash Network","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AKTUSD","name":"Akash - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AKT","base_currency_name":"Akash","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALBTUSD","name":"AllianceBlock - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALBT","base_currency_name":"AllianceBlock","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALCXUSD","name":"Alchemix - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALCX","base_currency_name":"Alchemix","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALEOUSD","name":"Aleo - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALEO","base_currency_name":"Aleo","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALEPHUSD","name":"Aleph.im v2 - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALEPH","base_currency_name":"Aleph.im v2","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALGOUSD","name":"Algorand - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALGO","base_currency_name":"Algorand","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALGUSD","name":"Algory - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALG","base_currency_name":"Algory","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALICEUSD","name":"MyNeighborAlice - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALICE","base_currency_name":"MyNeighborAlice","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALPHAUSD","name":"Stella - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALPHA","base_currency_name":"Stella","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALT11M2507USD","name":"ALT11M2507 - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALT11M2507","base_currency_name":"ALT11M2507","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALT11M250830USD","name":"ALT11M250830 - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALT11M250830","base_currency_name":"ALT11M250830","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALT11M251029USD","name":"ALT11M251029 - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALT11M251029","base_currency_name":"ALT11M251029","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALT2612USD","name":"ALT2612 - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALT2612","base_currency_name":"ALT2612","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ALTUSD","name":"AltLayer - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ALT","base_currency_name":"AltLayer","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AMPUSD","name":"Amp - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AMP","base_currency_name":"Amp","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ANKRUSD","name":"Ankr - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ANKR","base_currency_name":"Ankr","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ANTUSD","name":"Aragon - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ANT","base_currency_name":"Aragon","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:APENFTUSD","name":"APENFT - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"APENFT","base_currency_name":"APENFT","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:APEUSD","name":"ApeCoin - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"APE","base_currency_name":"ApeCoin","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:API3USD","name":"API3 - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"API3","base_currency_name":"API3","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:APPUSD","name":"Moon App - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"APP","base_currency_name":"Moon App","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:APTUSD","name":"Aptos - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"APT","base_currency_name":"Aptos","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:APUUSD","name":"APU - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"APU","base_currency_name":"APU","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ARBUSD","name":"Arbitrum - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ARB","base_currency_name":"Arbitrum","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ARKMUSD","name":"Arkham - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ARKM","base_currency_name":"Arkham","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ARPAUSD","name":"ARPA Chain - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ARPA","base_currency_name":"ARPA Chain","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ASMUSD","name":"Assemble Protocol - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ASM","base_currency_name":"Assemble Protocol","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ASTRUSD","name":"Astar - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ASTR","base_currency_name":"Astar","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ASTUSD","name":"AirSwap - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AST","base_currency_name":"AirSwap","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ATAUSD","name":"Automata Network - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ATA","base_currency_name":"Automata Network","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ATHUSD","name":"Aethir - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ATH","base_currency_name":"Aethir","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ATLASUSD","name":"Star Atlas - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ATLAS","base_currency_name":"Star Atlas","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ATOMUSD","name":"Cosmos - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ATOM","base_currency_name":"Cosmos","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:ATOUSD","name":"ATO - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"ATO","base_currency_name":"ATO","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AUCTIONUSD","name":"Bounce Token - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AUCTION","base_currency_name":"Bounce Token","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AUDIOUSD","name":"Audius - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AUDIO","base_currency_name":"Audius","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AURORAUSD","name":"Aurora - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AURORA","base_currency_name":"Aurora","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AUSDTUSD","name":"aUSDT - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AUSDT","base_currency_name":"aUSDT","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AVAXUSD","name":"Avalanche - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AVAX","base_currency_name":"Avalanche","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AVTUSD","name":"Aventus - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AVT","base_currency_name":"Aventus","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AXLUSD","name":"Axelar - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AXL","base_currency_name":"Axelar","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AXSUSD","name":"Axie Infinity - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AXS","base_currency_name":"Axie Infinity","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:AZEROUSD","name":"Aleph Zero - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"AZERO","base_currency_name":"Aleph Zero","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:B2MUSD","name":"Bit2Me - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"B2M","base_currency_name":"Bit2Me","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BADGERUSD","name":"Badger Dao - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BADGER","base_currency_name":"Badger Dao","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BALUSD","name":"Balancer - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BAL","base_currency_name":"Balancer","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BANDUSD","name":"Band Protocol - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BAND","base_currency_name":"Band Protocol","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BATUSD","name":"Basic Attention Token - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BAT","base_currency_name":"Basic Attention Token","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BCHEUR","name":"Bitcoin Cash - Euro","market":"crypto","locale":"global","active":true,"currency_symbol":"EUR","currency_name":"Euro","base_currency_symbol":"BCH","base_currency_name":"Bitcoin Cash","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BCHGBP","name":"Bitcoin Cash - Great Britian Pound","market":"crypto","locale":"global","active":true,"currency_symbol":"GBP","currency_name":"Great Britian Pound","base_currency_symbol":"BCH","base_currency_name":"Bitcoin Cash","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BCHNUSD","name":"Bitcoin Cash - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BCHN","base_currency_name":"Bitcoin Cash","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BCHUSD","name":"Bitcoin Cash - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BCH","base_currency_name":"Bitcoin Cash","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BEAMUSD","name":"BEAM - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BEAM","base_currency_name":"BEAM","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BESTUSD","name":"Bitpanda - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BEST","base_currency_name":"Bitpanda","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BFTUSD","name":"BnkToTheFuture - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BFT","base_currency_name":"BnkToTheFuture","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BGBUSD","name":"Bitget Token - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BGB","base_currency_name":"Bitget Token","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BICOUSD","name":"Biconomy - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BICO","base_currency_name":"Biconomy","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BIGTIMEUSD","name":"Big Time - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BIGTIME","base_currency_name":"Big Time","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BITUSD","name":"BitDAO - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BIT","base_currency_name":"BitDAO","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BLASTUSD","name":"Blast - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BLAST","base_currency_name":"Blast","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BLURUSD","name":"Blur - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BLUR","base_currency_name":"Blur","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BLZUSD","name":"Bluzelle - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BLZ","base_currency_name":"Bluzelle","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BNCUSD","name":"Bifrost Native Coin - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BNC","base_currency_name":"Bifrost Native Coin","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BNTUSD","name":"Bancor - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BNT","base_currency_name":"Bancor","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BOBAUSD","name":"Boba Network - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BOBA","base_currency_name":"Boba Network","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BODENUSD","name":"Jeo Boden - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BODEN","base_currency_name":"Jeo Boden","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BOMEUSD","name":"Book Of Meme - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BOME","base_currency_name":"Book Of Meme","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BONDUSD","name":"BarnBridge - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BOND","base_currency_name":"BarnBridge","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BONKEUR","name":"Bonk - Euro","market":"crypto","locale":"global","active":true,"currency_symbol":"EUR","currency_name":"Euro","base_currency_symbol":"BONK","base_currency_name":"Bonk","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BONKUSD","name":"Bonk - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BONK","base_currency_name":"Bonk","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BORGUSD","name":"BORG - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BORG","base_currency_name":"BORG","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BOSONUSD","name":"Boson Protocol - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BOSON","base_currency_name":"Boson Protocol","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BRICKUSD","name":"Brick by Brick - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BRICK","base_currency_name":"Brick by Brick","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BSVUSD","name":"Bitcoin SV - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BSV","base_currency_name":"Bitcoin SV","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BSXUSD","name":"Basilisk - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BSX","base_currency_name":"Basilisk","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BTCAUD","name":"Bitcoin - Australian Dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"AUD","currency_name":"Australian Dollar","base_currency_symbol":"BTC","base_currency_name":"Bitcoin","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BTCEUR","name":"Bitcoin - Euro","market":"crypto","locale":"global","active":true,"currency_symbol":"EUR","currency_name":"Euro","base_currency_symbol":"BTC","base_currency_name":"Bitcoin","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BTCGBP","name":"Bitcoin - Great Britain Pound","market":"crypto","locale":"global","active":true,"currency_symbol":"GBP","currency_name":"Great Britain Pound","base_currency_symbol":"BTC","base_currency_name":"Bitcoin","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BTCJPY","name":"Bitcoin - Japanese Yen","market":"crypto","locale":"global","active":true,"currency_symbol":"JPY","currency_name":"Japanese Yen","base_currency_symbol":"BTC","base_currency_name":"Bitcoin","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BTCPYUSD","name":"BTCpy - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BTCPY","base_currency_name":"BTCpy","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BTCUSD","name":"Bitcoin - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BTC","base_currency_name":"Bitcoin","last_updated_utc":"2017-01-01T00:00:00Z"},{"ticker":"X:BTGUSD","name":"Bitcoin Gold - United States dollar","market":"crypto","locale":"global","active":true,"currency_symbol":"USD","currency_name":"United States dollar","base_currency_symbol":"BTG","base_currency_name":"Bitcoin Gold","last_updated_utc":"2017-01-01T00:00:00Z"}],"status":"OK","request_id":"cbec3df2a7eab46127b2bd7f464384bb","count":100,"next_url":"https://api.polygon.io/v3/reference/tickers?cursor=YWN0aXZlPXRydWUmYXA9MTAwJmFzPSZsaW1pdD0xMDAmbWFya2V0PWNyeXB0byZvcmRlcj1hc2Mmc29ydD10aWNrZXI"}'''

In [49]:
import json
polygon_crypto_response = json.loads(polygon_crypto_assets)

In [59]:
pl.DataFrame(polygon_crypto_response['results'])

ticker,name,market,locale,active,currency_symbol,currency_name,base_currency_symbol,base_currency_name,last_updated_utc
str,str,str,str,bool,str,str,str,str,str
"""X:00USD""","""00 Token - United States dolla…","""crypto""","""global""",true,"""USD""","""United States dollar""","""00""","""00 Token""","""2017-01-01T00:00:00Z"""
"""X:1INCHUSD""","""1inch - United States dollar""","""crypto""","""global""",true,"""USD""","""United States dollar""","""1INCH""","""1inch""","""2017-01-01T00:00:00Z"""
"""X:A8USD""","""Ancient8 - United States dolla…","""crypto""","""global""",true,"""USD""","""United States dollar""","""A8""","""Ancient8""","""2017-01-01T00:00:00Z"""
"""X:AAVEUSD""","""Aave - United States dollar""","""crypto""","""global""",true,"""USD""","""United States dollar""","""AAVE""","""Aave""","""2017-01-01T00:00:00Z"""
"""X:ABTUSD""","""Arcblock - United States dolla…","""crypto""","""global""",true,"""USD""","""United States dollar""","""ABT""","""Arcblock""","""2017-01-01T00:00:00Z"""
…,…,…,…,…,…,…,…,…,…
"""X:BTCGBP""","""Bitcoin - Great Britain Pound""","""crypto""","""global""",true,"""GBP""","""Great Britain Pound""","""BTC""","""Bitcoin""","""2017-01-01T00:00:00Z"""
"""X:BTCJPY""","""Bitcoin - Japanese Yen""","""crypto""","""global""",true,"""JPY""","""Japanese Yen""","""BTC""","""Bitcoin""","""2017-01-01T00:00:00Z"""
"""X:BTCPYUSD""","""BTCpy - United States dollar""","""crypto""","""global""",true,"""USD""","""United States dollar""","""BTCPY""","""BTCpy""","""2017-01-01T00:00:00Z"""


In [55]:
polygon_crypto_list = set(pl.DataFrame(polygon_crypto_response['results']).select(pl.col('base_currency_symbol')).unique().to_series().to_list())
polygon_crypto_list

{'00',
 '1INCH',
 'A8',
 'AAVE',
 'ABT',
 'ACA',
 'ACH',
 'ACS',
 'ACX',
 'ADA',
 'ADX',
 'AERGO',
 'AERO',
 'AEVO',
 'AGLD',
 'AIOZ',
 'AIR',
 'AKT',
 'ALBT',
 'ALCX',
 'ALEO',
 'ALEPH',
 'ALG',
 'ALGO',
 'ALICE',
 'ALPHA',
 'ALT',
 'ALT11M2507',
 'ALT11M250830',
 'ALT11M251029',
 'ALT2612',
 'AMP',
 'ANKR',
 'ANT',
 'APE',
 'APENFT',
 'API3',
 'APP',
 'APT',
 'APU',
 'ARB',
 'ARKM',
 'ARPA',
 'ASM',
 'AST',
 'ASTR',
 'ATA',
 'ATH',
 'ATLAS',
 'ATO',
 'ATOM',
 'AUCTION',
 'AUDIO',
 'AURORA',
 'AUSDT',
 'AVAX',
 'AVT',
 'AXL',
 'AXS',
 'AZERO',
 'B2M',
 'BADGER',
 'BAL',
 'BAND',
 'BAT',
 'BCH',
 'BCHN',
 'BEAM',
 'BEST',
 'BFT',
 'BGB',
 'BICO',
 'BIGTIME',
 'BIT',
 'BLAST',
 'BLUR',
 'BLZ',
 'BNC',
 'BNT',
 'BOBA',
 'BODEN',
 'BOME',
 'BOND',
 'BONK',
 'BORG',
 'BOSON',
 'BRICK',
 'BSV',
 'BSX',
 'BTC',
 'BTCPY',
 'BTG'}

In [54]:
crypto_list = alpaca_crypto_list.intersection(polygon_crypto_list)
crypto_list

{'AAVE', 'AVAX', 'BAT', 'BCH', 'BTC'}

In [ ]:

TIMEFRAME_MAPPINGS = {
    '1Min': {'yahoo': '1m', 'alpaca': '1Min', 'polygon': ('minute', 1)},
    '2Min': {'yahoo': '2m', 'alpaca': '2Min', 'polygon': ('minute', 2)},
    '5Min': {'yahoo': '5m', 'alpaca': '5Min', 'polygon': ('minute', 5)},
    '10Min': {'yahoo': '10m', 'alpaca': '10Min', 'polygon': ('minute', 10)},
    '15Min': {'yahoo': '15m', 'alpaca': '15Min', 'polygon': ('minute', 15)},
    '30Min': {'yahoo': '30m', 'alpaca': '30Min', 'polygon': ('minute', 30)},
    '1Hour': {'yahoo': '60m', 'alpaca': '1Hour', 'polygon': ('hour', 1)},
    '4Hour': {'yahoo': '240m', 'alpaca': '4Hour', 'polygon': ('hour', 4)},
    '1Day': {'yahoo': '1d', 'alpaca': '1Day', 'polygon': ('day', 1)},
    '2Day': {'yahoo': '2d', 'alpaca': '2Day', 'polygon': ('day', 2)},
    '1Week': {'yahoo': '1wk', 'alpaca': '1Week', 'polygon': ('week', 1)},
    '2Week': {'yahoo': '2wk', 'alpaca': '2Week', 'polygon': ('week', 2)},
    '1Month': {'yahoo': '1mo', 'alpaca': '1Month', 'polygon': ('month', 1)},
    '3Month': {'yahoo': '3mo', 'alpaca': '3Month', 'polygon': ('month', 3)},
}

AVAILABLE_CRYPTO_ASSETS = ['AAVE',
                            'AVAX',
                            'BAT',
                            'BCH',
                            'BTC',
                            'CRV',
                            'DOGE',
                            'DOT',
                            'ETH',
                            'GRT',
                            'LINK',
                            'LTC',
                            'MKR',
                            'PEPE',
                            'SHIB',
                            'SOL',
                            'SUSHI',
                            'TRUMP',
                            'UNI',
                            'USDC',
                            'USDG',
                            'USDT',
                            'XRP',
                            'XTZ',
                            'YFI']

ALPACA_RESPONSE_CODES = {
    200: "Success",
    400: """One of the request parameters is invalid. See the returned message for details.""",
    403: """Authentication headers are missing or invalid. 
    Make sure you authenticate your request with a valid API key.""",
    429: """Too many requests. You hit the rate limit. 
    Use the X-RateLimit-... response headers to make sure you're under the rate limit.""",
    500: """Internal server error. We recommend retrying these later. 
    If the issue persists, please contact us on Slack or on the Community Forum."""
}

In [3]:
# backend/src/services/data_providers.py
from abc import ABC, abstractmethod
import polars as pl
from datetime import datetime, timedelta
import logging
from typing import Optional, Dict, Any, Tuple, Union, List
import yfinance as yf
import aiohttp
import asyncio
import pandas as pd

logger = logging.getLogger(__name__)

#### Base Data Provider Class

In [5]:

class BaseDataProvider(ABC):
    """Abstract base class for data providers"""
    
    @abstractmethod
    async def get_historical_data(
        self,
        symbol: str,
        start_date: datetime,
        end_date: datetime,
        timeframe: str
    ) -> pl.DataFrame:
        """Get historical OHLCV data"""
        pass
    
    @abstractmethod
    async def get_quote(self, symbol: str) -> Dict[str, Any]:
        """Get current quote for a symbol"""
        pass
    
    @abstractmethod
    async def get_crypto_quote(self, symbols: list[str]) -> pl.DataFrame:
        """Get crypto quote from Alpaca"""
        pass
    
    def get_provider_timeframe(self, timeframe: str) -> Union[str, Tuple[str, int], None]:
        """Get the provider-specific timeframe mapping"""
        provider_name = self.get_provider_name()
        if timeframe in TIMEFRAME_MAPPINGS:
            return TIMEFRAME_MAPPINGS[timeframe].get(provider_name)
        return None
    
    @abstractmethod
    def get_provider_name(self) -> str:
        """Return the provider name for timeframe mapping"""
        pass
    

#### YAHOO Finance Provider Class

In [ ]:

class YahooFinanceProvider(BaseDataProvider):
    """Yahoo Finance data provider"""
    
    def get_provider_name(self) -> str:
        return 'yahoo'
    
    async def get_historical_data(
        self,
        symbol: str,
        start_date: datetime,
        end_date: datetime,
        timeframe: str
    ) -> pl.DataFrame:
        """Get historical data from Yahoo Finance"""
        # Run in thread pool to avoid blocking
        loop = asyncio.get_event_loop()
        
        def fetch_data():
            ticker = yf.Ticker(symbol)
            interval = self.get_provider_timeframe(timeframe) or '1d'
            
            df = ticker.history(
                start=start_date,
                end=end_date,
                interval=interval,
                auto_adjust=True
            )
            
            # Rename columns to lowercase
            df.columns = df.columns.str.lower()
            
            # Ensure we have all required columns
            required_columns = ['open', 'high', 'low', 'close', 'volume']
            for col in required_columns:
                if col not in df.columns:
                    df[col] = 0
            
            return df
        
        return await loop.run_in_executor(None, fetch_data)
    
    async def get_quote(self, symbol: str) -> Dict[str, Any]:
        """Get current quote from Yahoo Finance"""
        loop = asyncio.get_event_loop()
        
        def fetch_quote():
            ticker = yf.Ticker(symbol)
            info = ticker.info
            
            return {
                'symbol': symbol,
                'price': info.get('regularMarketPrice', 0),
                'bid': info.get('bid', 0),
                'ask': info.get('ask', 0),
                'volume': info.get('regularMarketVolume', 0),
                'timestamp': datetime.now()
            }
        
        return await loop.run_in_executor(None, fetch_quote)


#### Alpaca Data Provider Class

In [ ]:

class AlpacaProvider(BaseDataProvider):
    """Alpaca Markets data provider"""
    
    def __init__(self, api_key: str, secret_key: str, base_url: str = 'https://data.alpaca.markets/v2'):
        self.api_key = api_key
        self.secret_key = secret_key
        self.base_url = base_url
        self.headers = {
            "APCA-API-KEY-ID": self.api_key,
            "APCA-API-SECRET-KEY": self.secret_key,
            "Content-Type": "application/json"
        }
    
    def get_provider_name(self) -> str:
        return 'alpaca'
    
    async def get_historical_data(self,
                                  symbols: List[str],
                                  start_date: datetime,
                                  end_date: datetime,
                                  timeframe: str
                                  ) -> pl.DataFrame:
        """
        Fetches historical bar data for multiple symbols from Alpaca, handling pagination.

        Args:
            symbols: A list of stock symbols.
            start_date: The start date for the historical data.
            end_date: The end date for the historical data.
            timeframe: The timeframe for the bars (e.g., '1D', '1H', '1Min').

        Returns:
            A Polars DataFrame containing the historical data with a 'symbol' column.
            Returns an empty DataFrame if no data is found or an error occurs.
        """
        async with aiohttp.ClientSession() as session:
            timeframe_str = self.get_provider_timeframe(timeframe) or '1Day'
            url = f"{self.base_url}/v2/stocks/bars"
            
            params = {
                'symbols': ','.join(symbols),
                'start': start_date.isoformat(),
                'end': end_date.isoformat(),
                'timeframe': timeframe_str,
                'limit': 10000,
                'adjustment': 'raw'
            }
            
            all_bars = []
            page_token = None
            
            while True:
                if page_token:
                    params['page_token'] = page_token
                
                try:
                    async with session.get(url, headers=self.headers, params=params) as response:
                        response.raise_for_status()

                        if response.status == 200:
                            data = await response.json()
                            bars_data = data.get('bars')
                            if bars_data:
                                for symbol, symbol_bars in bars_data.items():
                                    for bar in symbol_bars:
                                        bar['symbol'] = symbol
                                        all_bars.append(bar)
                            page_token = data.get('next_page_token')
                            if not page_token:
                                break
                        else:
                            logger.error(f"{ALPACA_RESPONSE_CODES[response.status]}")
                            return pl.DataFrame()
                        
                except aiohttp.ClientError as e:
                    logger.error(f"Error fetching historical data from Alpaca: {e}")
                    return pl.DataFrame()
            
            if not all_bars:
                return pl.DataFrame()

            df = pl.DataFrame(all_bars)

            df = df.with_columns(
                pl.col("t").str.to_datetime().alias("timestamp")
            ).rename({
                'o': 'open',
                'h': 'high',
                'l': 'low',
                'c': 'close',
                'v': 'volume',
                'vw': 'vwap'
            })
            
            # Ensure all required columns are present before selecting
            required_cols = ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap']
            
            # Filter out columns that are not in the DataFrame
            existing_cols = [col for col in required_cols if col in df.columns]
            
            return df.select(existing_cols)
    
    async def get_quote(self, symbol: str) -> Dict[str, Any]:
        """Get current quote from Alpaca"""
        async with aiohttp.ClientSession() as session:
            url = f"{self.base_url}/v2/stocks/{symbol}/quotes/latest"
            
            async with session.get(url, headers=self.headers) as response:
                data = await response.json()
                
                if 'quote' in data:
                    quote = data['quote']
                    return {
                        'symbol': symbol,
                        'price': quote.get('ap', 0),  # ask price
                        'bid': quote.get('bp', 0),     # bid price
                        'ask': quote.get('ap', 0),     # ask price
                        'volume': quote.get('as', 0),  # ask size
                        'timestamp': pl.to_datetime(quote.get('t'))
                    }
                else:
                    return {}

    async def get_crypto_historical_data(self,
                                         symbols: list[str],
                                         start_date: datetime,
                                         end_date: datetime,
                                         timeframe: str
                                         ) -> pl.DataFrame:
        """
        Fetches historical bar data for multiple symbols from Alpaca, handling pagination.

        Args:
            symbols: A list of stock symbols.
            start_date: The start date for the historical data.
            end_date: The end date for the historical data.
            timeframe: The timeframe for the bars (e.g., '1D', '1H', '1Min').

        Returns:
            A Polars DataFrame containing the historical data with a 'symbol' column.
            Returns an empty DataFrame if no data is found or an error occurs.
        """
        for symbol in symbols:
            if symbol not in AVAILABLE_CRYPTO_ASSETS:
                raise ValueError(f"Symbol {symbol} is not a tradable crypto asset on Alpaca")
        
        symbols = ['/USD'.join(x) for x in symbols]

        async with aiohttp.ClientSession() as session:
            url = f"https://data.alpaca.markets/v1beta3/crypto/us/bars"
            params = {
                'symbols': symbols,
                'start': start_date.isoformat(),
                'end': end_date.isoformat(),
                'timeframe': timeframe,
                'limit': 10000,
                'adjustment': 'raw'
            }
            all_bars = []
            page_token = None
            
            while True:
                if page_token:
                    params['page_token'] = page_token
                
                try:
                    async with session.get(url, headers=self.headers, params=params) as response:
                        response.raise_for_status()

                        if response.status == 200:
                            data = await response.json()
                            bars_data = data.get('bars')
                            if bars_data:
                                for symbol, symbol_bars in bars_data.items():
                                    for bar in symbol_bars:
                                        bar['symbol'] = symbol
                                        all_bars.append(bar)
                            page_token = data.get('next_page_token')
                            if not page_token:
                                break
                        else:
                            logger.error(f"{ALPACA_RESPONSE_CODES[response.status]}")
                            return pl.DataFrame()
                        
                except aiohttp.ClientError as e:
                    logger.error(f"Error fetching historical data from Alpaca: {e}")
                    return pl.DataFrame()
            
            if not all_bars:
                return pl.DataFrame()

            df = pl.DataFrame(all_bars)

            df = df.with_columns(
                pl.col("t").str.to_datetime().alias("timestamp")
            ).rename({
                'o': 'open',
                'h': 'high',
                'l': 'low',
                'c': 'close',
                'v': 'volume',
                'vw': 'vwap'
            })
            
            # Ensure all required columns are present before selecting
            required_cols = ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap']
            
            # Filter out columns that are not in the DataFrame
            existing_cols = [col for col in required_cols if col in df.columns]
            
            return df.select(existing_cols)

    async def get_asset_list(self):
        """Get list of assets from Alpaca"""
        async with aiohttp.ClientSession() as session:
            url = "https://paper-api.alpaca.markets/v2/assets"
            async with session.get(url, headers=self.headers) as response:
                data = await response.json()
                return data

    async def get_crypto_quote(self, symbols: list[str]) -> pl.DataFrame:
        """Get crypto quote from Alpaca"""

        for symbol in symbols:
            if symbol not in AVAILABLE_CRYPTO_ASSETS:
                raise ValueError(f"Symbol {symbol} is not a tradable crypto asset on Alpaca")

        symbols = ['/USD'.join(x) for x in symbols]

        async with aiohttp.ClientSession() as session:
            base_url = "https://data.alpaca.markets/v1beta3/crypto/us/latest/bars"
            params = {
                'symbols': symbols
            }
            async with session.get(base_url, params=params) as response:
                data = await response.json()
                data = pl.DataFrame(data['bars'])
                return data

#### Polygon Provider Class

In [ ]:

class PolygonProvider(BaseDataProvider):
    """Polygon.io data provider"""
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = 'https://api.polygon.io'
    
    def get_provider_name(self) -> str:
        return 'polygon'
    
    async def get_historical_data(self,
                                  symbol: str,
                                  start_date: datetime,
                                  end_date: datetime,
                                  timeframe: str
                                  ) -> pl.DataFrame:
        """
        Get historical data from Polygon
        Args:
            symbol: str
            start_date: datetime
            end_date: datetime
            timeframe: str
        Returns:
            pl.DataFrame
        Raises:
            ValueError: If the symbol is not a valid crypto or stock asset on Polygon
        """
        timespan_multiplier = TIMEFRAME_MAPPINGS[timeframe].get('polygon')
        if timespan_multiplier is None:
            raise ValueError(f"Timeframe {timeframe} is not supported by Polygon")
        else:
            timespan = timespan_multiplier[0]
            multiplier = timespan_multiplier[1]

        if symbol not in AVAILABLE_CRYPTO_ASSETS:
            url = f"{self.base_url}/v2/aggs/ticker/{symbol}/range/{multiplier}/{timespan}/{start_date}/{end_date}"
        else:
            crypto_symbol = 'X:' + symbol + 'USD'
            url = f"{self.base_url}/v2/aggs/ticker/{crypto_symbol}/range/{multiplier}/{timespan}/{start_date}/{end_date}"

        params = {
            'apiKey': self.api_key,
            'adjusted': 'true',
            'sort': 'asc',
            'limit': 50000
        }


        async with aiohttp.ClientSession() as session:
            async with session.get(url, params=params) as response:
                data = await response.json()
                if response.status == 200:      
                    logger.info(f"Data Provider: Polygon Response: {response.status}")
                else:
                    logger.error(f"Data Provider: Polygon Response: {data}")

                if 'results' in data and data['results']:
                    df = pl.DataFrame(data['results'])
                    
                    # Convert timestamp to datetime
                    df = df.with_columns(
                        pl.from_epoch('t', time_unit='ms').dt.convert_time_zone("America/New_York").alias("t")
                    )
                    
                    # Rename columns
                    df = df.rename({
                        'o': 'open',
                        'h': 'high',
                        'l': 'low',
                        'c': 'close',
                        'v': 'volume',
                        'vw': 'vwap'
                    })
                    
                    return df[['t', 'open', 'high', 'low', 'close', 'volume', 'vwap']]
                else:
                    return pl.DataFrame()
                
    def _create_empty_bar_record(self, symbol: str) -> Dict[str, Any]:
        """Create an empty bar record for symbols with no data"""
        return {
            'symbol': symbol,
            'timestamp': 0,
            'open': 0.0,
            'high': 0.0,
            'low': 0.0,
            'close': 0.0,
            'volume': 0.0,
            'vwap': 0.0,
            'transactions': 0,
            'bid_estimate': 0.0,
            'ask_estimate': 0.0,
            'last_updated_utc': datetime.now().isoformat()
        }
    
    async def get_crypto_quote(self, symbols: list[str]) -> pl.DataFrame:
        """
        Get crypto quotes from Polygon using the latest 1-minute bars.
        This provides near real-time OHLCV data which is more comprehensive than quotes.
        
        Args:
            symbols: List of crypto symbols (e.g., ['BTC', 'ETH', 'SOL'])
                    Note: These should be base currency symbols, USD will be appended
        
        Returns:
            pl.DataFrame with columns: [symbol, timestamp, open, high, low, close, 
                                    volume, vwap, transactions, bid_estimate, ask_estimate]
            Returns empty DataFrame if no data found or errors occur
        
        Raises:
            ValueError: If any symbol is not in AVAILABLE_CRYPTO_ASSETS
        """
        # Validate all symbols first
        for symbol in symbols:
            if symbol not in AVAILABLE_CRYPTO_ASSETS:
                raise ValueError(f"Symbol {symbol} is not a tradable crypto asset on Polygon")
        
        if not symbols:
            logger.warning("No symbols provided to get_crypto_quote")
            return pl.DataFrame()
        
        # Calculate time range for latest bars (last 2 minutes to ensure we get data)
        end_time = datetime.now()
        start_time = end_time - timedelta(minutes=2)
        
        # Format dates for API
        from_date = start_time.strftime('%Y-%m-%d')
        to_date = end_time.strftime('%Y-%m-%d')
        
        async with aiohttp.ClientSession() as session:
            all_bars = []
            
            # Use semaphore to limit concurrent requests (respect rate limits)
            semaphore = asyncio.Semaphore(5)
            
            async def fetch_symbol_bar(symbol: str):
                async with semaphore:
                    try:
                        # Polygon crypto format: X:SYMBOLUSD
                        crypto_ticker = f"X:{symbol}USD"
                        
                        # Get latest 1-minute bars for near real-time data
                        url = f"{self.base_url}/v2/aggs/ticker/{crypto_ticker}/range/1/minute/{from_date}/{to_date}"
                        params = {
                            'apikey': self.api_key,
                            'adjusted': 'true',
                            'sort': 'desc',  # Get latest bars first
                            'limit': 1       # Only need the most recent bar
                        }
                        
                        async with session.get(url, params=params) as response:
                            if response.status == 200:
                                data = await response.json()
                                logger.info(f"Polygon crypto bar for {symbol}: Status {response.status}")
                                
                                if 'results' in data and data['results']:
                                    # Get the most recent bar (first one due to desc sort)
                                    latest_bar = data['results'][0]
                                    
                                    # Parse bar data
                                    bar_record = {
                                        'symbol': symbol,
                                        'timestamp': latest_bar.get('t', 0),  # Unix timestamp in ms
                                        'open': latest_bar.get('o', 0.0),
                                        'high': latest_bar.get('h', 0.0),
                                        'low': latest_bar.get('l', 0.0),
                                        'close': latest_bar.get('c', 0.0),     # This is our "current price"
                                        'volume': latest_bar.get('v', 0.0),
                                        'vwap': latest_bar.get('vw', 0.0),     # Volume weighted average price
                                        'transactions': latest_bar.get('n', 0), # Number of transactions
                                        'last_updated_utc': datetime.now().isoformat()
                                    }
                                    
                                    # Estimate bid/ask from OHLC (common practice)
                                    # Bid = slightly below close, Ask = slightly above close
                                    close_price = bar_record['close']
                                    if close_price > 0:
                                        spread_estimate = close_price * 0.001  # 0.1% spread estimate
                                        bar_record['bid_estimate'] = close_price - (spread_estimate / 2)
                                        bar_record['ask_estimate'] = close_price + (spread_estimate / 2)
                                    else:
                                        bar_record['bid_estimate'] = 0.0
                                        bar_record['ask_estimate'] = 0.0
                                    
                                    return bar_record
                                
                                else:
                                    logger.warning(f"No bar data found for {symbol}")
                                    return self._create_empty_bar_record(symbol)
                            
                            elif response.status == 401:
                                logger.error("Polygon API authentication failed - check API key")
                                return None
                            
                            elif response.status == 403:
                                logger.error("Polygon API access forbidden - check subscription plan")
                                return None
                            
                            elif response.status == 429:
                                logger.error(f"Polygon API rate limit exceeded for {symbol}")
                                # Return empty record rather than None to continue processing
                                return self._create_empty_bar_record(symbol)
                            
                            elif response.status == 404:
                                logger.warning(f"No data found for crypto symbol {symbol}")
                                return self._create_empty_bar_record(symbol)
                            
                            else:
                                logger.error(f"Polygon API error for {symbol}: Status {response.status}")
                                response_text = await response.text()
                                logger.error(f"Response: {response_text}")
                                return self._create_empty_bar_record(symbol)
                    
                    except aiohttp.ClientError as e:
                        logger.error(f"Network error fetching bar for {symbol}: {e}")
                        return self._create_empty_bar_record(symbol)
                    
                    except Exception as e:
                        logger.error(f"Unexpected error fetching bar for {symbol}: {e}")
                        return self._create_empty_bar_record(symbol)
            
            # Execute all requests concurrently
            tasks = [fetch_symbol_bar(symbol) for symbol in symbols]
            results = await asyncio.gather(*tasks, return_exceptions=True)
            
            # Filter out None results and exceptions
            for result in results:
                if result is not None and not isinstance(result, Exception):
                    all_bars.append(result)
            
            # Convert to Polars DataFrame
            if not all_bars:
                logger.warning("No crypto bars retrieved from Polygon")
                return pl.DataFrame()
            
            try:
                df = pl.DataFrame(all_bars)
                
                # Convert timestamp and ensure proper data types
                df = df.with_columns([
                    # Convert timestamp from milliseconds to datetime
                    pl.when(pl.col("timestamp") > 0)
                    .then(pl.from_epoch("timestamp", time_unit="ms").dt.convert_time_zone("UTC"))
                    .otherwise(pl.lit(None).cast(pl.Datetime))
                    .alias("timestamp"),
                    
                    # Ensure numeric columns are proper types
                    pl.col("open").cast(pl.Float64),
                    pl.col("high").cast(pl.Float64),
                    pl.col("low").cast(pl.Float64),
                    pl.col("close").cast(pl.Float64),
                    pl.col("volume").cast(pl.Float64),
                    pl.col("vwap").cast(pl.Float64),
                    pl.col("transactions").cast(pl.Int32),
                    pl.col("bid_estimate").cast(pl.Float64),
                    pl.col("ask_estimate").cast(pl.Float64),
                ])
                
                # Add additional calculated fields for compatibility
                df = df.with_columns([
                    # Mid price (same as close for bars)
                    pl.col("close").alias("mid_price"),
                    
                    # Price change from open to close
                    pl.when(pl.col("open") > 0)
                    .then(pl.col("close") - pl.col("open"))
                    .otherwise(0.0)
                    .alias("price_change"),
                    
                    # Percentage change
                    pl.when(pl.col("open") > 0)
                    .then(((pl.col("close") - pl.col("open")) / pl.col("open")) * 100)
                    .otherwise(0.0)
                    .alias("price_change_percent"),
                    
                    # Trading intensity (volume per transaction)
                    pl.when(pl.col("transactions") > 0)
                    .then(pl.col("volume") / pl.col("transactions"))
                    .otherwise(0.0)
                    .alias("avg_trade_size")
                ])
                
                # Select and order columns for final output
                final_columns = [
                    'symbol', 'timestamp', 'close', 'open', 'high', 'low', 
                    'volume', 'vwap', 'transactions', 'bid_estimate', 'ask_estimate',
                    'mid_price', 'price_change', 'price_change_percent', 'avg_trade_size',
                    'last_updated_utc'
                ]
                
                # Filter columns that exist in the DataFrame
                existing_columns = [col for col in final_columns if col in df.columns]
                
                return df.select(existing_columns)
                
            except Exception as e:
                logger.error(f"Error creating DataFrame from crypto bars: {e}")
                return pl.DataFrame()

    async def get_quote(self, symbol: str) -> Dict[str, Any]:
        """Get current quote from Polygon using latest 1-minute bar"""
        async with aiohttp.ClientSession() as session:
            # Get the most recent 1-minute bar (last 2 minutes to ensure data)
            end_date = datetime.now().strftime('%Y-%m-%d')
            start_date = (datetime.now() - timedelta(minutes=2)).strftime('%Y-%m-%d')
            
            url = f"{self.base_url}/v2/aggs/ticker/{symbol}/range/1/minute/{start_date}/{end_date}"
            params = {
                'apikey': self.api_key,
                'adjusted': 'true',
                'sort': 'desc',
                'limit': 1
            }
            
            async with session.get(url, params=params) as response:
                data = await response.json()
                
                if 'results' in data and data['results']:
                    bar = data['results'][0]  # Most recent bar
                    close_price = bar.get('c', 0)
                    
                    # Simple bid/ask estimation from close price
                    spread = close_price * 0.001  # 0.1% spread estimate
                    bid_price = close_price - (spread / 2)
                    ask_price = close_price + (spread / 2)
                    
                    return {
                        'symbol': symbol,
                        'price': close_price,
                        'bid': bid_price,
                        'ask': ask_price,
                        'volume': bar.get('v', 0),
                        'timestamp': pl.from_epoch(bar.get('t', 0), time_unit='ms')
                    }
                else:
                    return {}
                

class DataProviderFactory:
    """Factory class to create data providers"""
    
    @staticmethod
    def get_provider(
        provider_name: str,
        **kwargs
    ) -> BaseDataProvider:
        """Get a data provider instance"""
        provider_name = provider_name.lower()
        
        if provider_name == 'yahoo':
            return YahooFinanceProvider()
        
        elif provider_name == 'alpaca':
            if 'api_key' not in kwargs or 'secret_key' not in kwargs:
                raise ValueError("Alpaca provider requires api_key and secret_key")
            return AlpacaProvider(
                api_key=kwargs['api_key'],
                secret_key=kwargs['secret_key'],
                base_url=kwargs.get('base_url', 'https://data.alpaca.markets')
            )
        
        elif provider_name == 'polygon':
            if 'api_key' not in kwargs:
                raise ValueError("Polygon provider requires api_key")
            return PolygonProvider(api_key=kwargs['api_key'])
        
        else:
            raise ValueError(f"Unknown data provider: {provider_name}")
    
    @staticmethod
    def get_supported_timeframes(provider_name: str = None) -> Dict[str, list]:
        """Get supported timeframes for a specific provider or all providers"""
        if provider_name:
            provider_name = provider_name.lower()
            if provider_name not in ['yahoo', 'alpaca', 'polygon']:
                raise ValueError(f"Unknown provider: {provider_name}")
            
            supported = []
            for timeframe, mappings in TIMEFRAME_MAPPINGS.items():
                if mappings.get(provider_name) is not None:
                    supported.append(timeframe)
            return {provider_name: supported}
        else:
            # Return all supported timeframes for each provider
            result = {'yahoo': [], 'alpaca': [], 'polygon': []}
            for timeframe, mappings in TIMEFRAME_MAPPINGS.items():
                for provider in result.keys():
                    if mappings.get(provider) is not None:
                        result[provider].append(timeframe)
            return result

In [9]:
l1 = ['list']
','.join(l1)

'list'

In [15]:
import json

TypeError: unexpected value while building Series of type Float64; found value of type Int64: 185

Hint: Try setting `strict=False` to allow passing data with mixed types.

In [23]:
import requests

url = "https://data.alpaca.markets/v2/stocks/bars?symbols=AAPL%2CTSLA&timeframe=1H&start=2024-01-03&end=2024-01-04&limit=1000&adjustment=raw&feed=sip&sort=asc"

headers = {
    "accept": "application/json",
    "APCA-API-KEY-ID": "PKW94MLMTLDYGACJ53PJ",
    "APCA-API-SECRET-KEY": "Ewzw0ngdmRtT0W4R1JqTVBOfezkEq5d92SKIznOO"
}

response = requests.get(url, headers=headers)

print(response.text)

{"bars":{"AAPL":[{"c":185.2,"h":185.31,"l":185.15,"n":1676,"o":185.31,"t":"2024-01-03T00:00:00Z","v":57455,"vw":185.239105},{"c":184.72,"h":185,"l":184.72,"n":960,"o":185,"t":"2024-01-03T09:00:00Z","v":27324,"vw":184.896828},{"c":184.73,"h":184.83,"l":184.67,"n":467,"o":184.67,"t":"2024-01-03T10:00:00Z","v":17545,"vw":184.780796},{"c":185,"h":185,"l":184.66,"n":682,"o":184.8,"t":"2024-01-03T11:00:00Z","v":21380,"vw":184.804194},{"c":184.31,"h":184.99,"l":184.31,"n":2988,"o":184.95,"t":"2024-01-03T12:00:00Z","v":89306,"vw":184.650556},{"c":184.65,"h":185.64,"l":184.19,"n":7701,"o":184.975,"t":"2024-01-03T13:00:00Z","v":395721,"vw":184.684691},{"c":184.7701,"h":185.88,"l":183.5,"n":104617,"o":184.68,"t":"2024-01-03T14:00:00Z","v":9882363,"vw":184.933103},{"c":183.86,"h":185.29,"l":183.45,"n":206132,"o":184.77,"t":"2024-01-03T15:00:00Z","v":9495059,"vw":184.042334},{"c":183.51,"h":184.38,"l":183.49,"n":88674,"o":183.855,"t":"2024-01-03T16:00:00Z","v":5895725,"vw":183.979655},{"c":184.335,

In [ ]:
x